In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [2]:
import sys
print(sys.executable)

from youtube_transcript_api import YouTubeTranscriptApi
print("youtube-transcript-api OK")

from langchain_text_splitters import RecursiveCharacterTextSplitter
print("text splitter OK")

from langchain_google_genai import GoogleGenerativeAIEmbeddings
print("gemini OK")

d:\backup desk\Langchain\YoutubeChatbot\.venv\Scripts\python.exe
youtube-transcript-api OK
text splitter OK
gemini OK


In [14]:
from utils import extract_video_id

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi ,TranscriptsDisabled

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (

GoogleGenerativeAIEmbeddings,

ChatGoogleGenerativeAI

)

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

C:\Users\nikhi\AppData\Local\Temp\ipykernel_11408\184363516.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
#Step 1a - Indexing (Document Ingestion)

In [16]:
def validate_youtube_url(url):

    video_id = extract_video_id(url)

    if video_id is None:
        return False

    return True

In [ ]:
url = "https://www.youtube.com/watch?v=gbeXe_o-Xf4"
#raj shamani podcast with perplexity ceo
#video_id = url.split("v=")[1].split("&")[0]
#video_id = extract_video_id(url)
if not validate_youtube_url(url):
    print("❌ Invalid YouTube URL")
else:
    video_id = extract_video_id(url)
    print("Video ID :", video_id)


Video ID : gbeXe_o-Xf4


In [5]:
#video_id = "J5_-l7WIO_w" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
   from youtube_transcript_api import YouTubeTranscriptApi

   ytt_api = YouTubeTranscriptApi()

   transcript = ytt_api.fetch(video_id, languages=["en"])


   text = " ".join([snippet.text for snippet in transcript])

   print(text)

except TranscriptsDisabled:
    print("No captions available for this video.")

You You've said in your research that in your practices and in your work that before you we sleep, that time is really important. And if we practice gratitude 10 minutes a day, in just about 4 days our immunity system gets 50% better. Yeah. Is that it? So then that is one case. And then a lot of people just use Instagram for 10 minutes [laughter] by the end of the day, right? What's If I grat- If I practice gratitude before bed for 10 minutes versus I use my Instagram for 10 minutes, what's going to be the difference? >> Some people can't think for themselves because they're having something think for them, right? Those images and the stimulation is causing them to really expose their brain to information um that is causing them to think and feel a certain way. What I'm saying is think and feel on your own, right? And that's the most important thing or think for yourself. So So people who lose the imagination or the skill of dreaming, uh they're just relying on something to think for t

In [6]:
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
)

def get_transcript(video_id):
    ytt_api = YouTubeTranscriptApi()

    try:
        transcript_list = ytt_api.list(video_id)

        try:
            transcript = transcript_list.find_transcript(
                ["en", "hi", "mr", "ta", "te", "kn", "ml", "bn", "gu"]
            )
        except NoTranscriptFound:
            transcript = transcript_list.find_generated_transcript(
                ["en", "hi", "mr", "ta", "te", "kn", "ml", "bn", "gu"]
            )

        print("Language:", transcript.language)

        # Uncomment if you want all text in English
        # transcript = transcript.translate("en")

        text = " ".join(snippet.text for snippet in transcript.fetch())
        return text

    except TranscriptsDisabled:
        return "Captions are disabled for this video."

In [ ]:
##- Indexing (Text Splitting)

In [21]:
splitter = RecursiveCharacterTextSplitter( chunk_size=800,
    chunk_overlap=150)
#chunks = splitter.create_documents([transcript])

# If transcript is already a string
chunks = splitter.create_documents([str(transcript)])

In [22]:
len(chunks)

27

In [ ]:
## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [23]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
     temperature=0
)

vector_store = FAISS.from_documents(chunks, embeddings)  #FAISS VECTOR  EMBEDINGS STORE

In [ ]:
## Step 2 - Retrieval

In [9]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
## Step 3 - Augmentation

In [40]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.2
)

In [26]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [20]:


prompt = PromptTemplate(
    template="""
You are a helpful YouTube AI Assistant.

You MUST answer ONLY using the transcript provided below.

Rules:
1. Never make up information.
2. If the answer is not present in the transcript, reply:
   "I couldn't find that information in the video."
3. Keep answers clear and concise.
4. If possible, answer using bullet points.
5. If the question is unrelated to the transcript, politely say that it is outside the video's content.

Transcript:
{context}

Question:
{input}

Answer:
""",
input_variables = ['context', 'question']
)

In [27]:
question          = "what is topic and give five points"
retrieved_docs    = retriever.invoke(question)

In [13]:
retrieved_docs

[Document(id='2aa42c84-2944-414a-aa7c-e8026a5ba3eb', metadata={}, page_content='FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="You You\'ve said in your research that", start=2.68, duration=6.36), FetchedTranscriptSnippet(text=\'in your practices and in your work\', start=6.28, duration=4.76), FetchedTranscriptSnippet(text=\'that before you we sleep, that time is\', start=9.04, duration=4.68), FetchedTranscriptSnippet(text=\'really important. And if we practice\', start=11.04, duration=5.32), FetchedTranscriptSnippet(text=\'gratitude 10 minutes a day, in just\', start=13.72, duration=5.04), FetchedTranscriptSnippet(text=\'about 4 days our immunity system gets\', start=16.36, duration=6.0), FetchedTranscriptSnippet(text=\'50% better. Yeah. Is that it? So then\', start=18.76, duration=5.48), FetchedTranscriptSnippet(text=\'that is one case. And then a lot of\', start=22.36, duration=3.84), FetchedTranscriptSnippet(text=\'people just use Instagram for 10 minutes\', start=24.24,

In [28]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="You You\'ve said in your research that", start=2.68, duration=6.36), FetchedTranscriptSnippet(text=\'in your practices and in your work\', start=6.28, duration=4.76), FetchedTranscriptSnippet(text=\'that before you we sleep, that time is\', start=9.04, duration=4.68), FetchedTranscriptSnippet(text=\'really important. And if we practice\', start=11.04, duration=5.32), FetchedTranscriptSnippet(text=\'gratitude 10 minutes a day, in just\', start=13.72, duration=5.04), FetchedTranscriptSnippet(text=\'about 4 days our immunity system gets\', start=16.36, duration=6.0), FetchedTranscriptSnippet(text=\'50% better. Yeah. Is that it? So then\', start=18.76, duration=5.48), FetchedTranscriptSnippet(text=\'that is one case. And then a lot of\', start=22.36, duration=3.84), FetchedTranscriptSnippet(text=\'people just use Instagram for 10 minutes\', start=24.24, duration=4.32), FetchedTranscriptSnippet(text=\'[laughter] by the end of the d

In [29]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
## Step 4 - Generation

In [41]:
answer = llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': 'The topic is the importance of the time before sleep and upon waking, specifically regarding how brain waves and the subconscious mind function during these periods.\n\nHere are five points based on the transcript:\n\n1.  **Brain Wave Transitions:** In the evening, brain waves transition from beta to alpha, theta, and delta; in the morning, they transition from delta to theta, alpha, and beta.\n2.  **The "Door" Opening:** These two times of day are when the "door" between the conscious and subconscious mind opens.\n3.  **Chemical Changes:** When going to bed, serotonin levels change to melatonin, which facilitates the shift into alpha, theta, and delta states.\n4.  **Intentional Practice:** Because the subconscious is accessible during these times, it is recommended to use the time before bed to write down things that are important or things one wants to change, rather than watching TV or using devices.\n5.  **Health Benefits:** Practicing gratitude for 10 mi

In [ ]:
## Building a Chain

In [42]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [43]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs) #concatenate all string documents
  return context_text

In [44]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [45]:
parallel_chain.invoke('what is habits ')

{'context': 'So, there\'s two", start=286.32, duration=4.44), FetchedTranscriptSnippet(text=\'times uh where the door between the\', start=288.4, duration=3.88), FetchedTranscriptSnippet(text=\'conscious mind and subconscious mind\', start=290.76, duration=2.6), FetchedTranscriptSnippet(text=\'opens.\', start=292.28, duration=3.48), FetchedTranscriptSnippet(text=\'And people I mean, the people in my life\', start=293.36, duration=5.52), FetchedTranscriptSnippet(text=\'that that do this, they they they they\', start=295.76, duration=5.04), FetchedTranscriptSnippet(text=\'take their time before they go to bed.\', start=298.88, duration=3.56), FetchedTranscriptSnippet(text=\'They write things down that are\', start=300.8, duration=3.48), FetchedTranscriptSnippet(text=\'important to them, things they want to\', start=302.44, duration=4.2), FetchedTranscriptSnippet(text="change. Uh they don\'t want to watch TV", start=304.28, duration=4.2), FetchedTranscriptSnippet(text=\'or get on their th

In [46]:
parser = StrOutputParser()

In [47]:
main_chain = parallel_chain | prompt | llm | parser

In [48]:
main_chain.invoke('Can you summarize the video')

'Based on the provided transcript, the video discusses the importance of the time before sleep and how it impacts the brain and body. Key points include:\n\n*   **Gratitude and Immunity:** Practicing gratitude for 10 minutes a day can improve the immune system by 50% in about four days.\n*   **Impact of Stimulation:** Using platforms like Instagram before bed exposes the brain to stimulation and information that influences how people think and feel, which can hinder the ability to think for oneself or dream.\n*   **Biological Changes Before Sleep:** When going to bed, serotonin levels change to melatonin. This shift moves the brain from a beta state into alpha, theta, and delta states, which is described as a time when "the door opens."\n*   **Personal Agency:** The speaker emphasizes the importance of thinking for oneself and suggests that individuals are the creators of their own lives.'